Be carefull to change the class in the videoMAE repository modeling_pretrain (modification in the kwarg to remove init_ckpt in the beginning and use it later), without this change the model doeesn't load the checkpoint 
also change the parameters depending of the parameters set during the training

In [7]:
import sys
sys.path.append("./VideoMAE")  # directory containing both modeling_pretrain.py and modeling_finetune.py
from modeling_pretrain import pretrain_videomae_base_patch16_224

# Load the model and checkpoint
model = pretrain_videomae_base_patch16_224(pretrained=True, init_ckpt="/home/unruffled_satoshi/workdir/PFR-ViTCow/videomae_training/results/testing_1GPU_1EPOCH/checkpoint-0.pth")

# Freeze encoder
encoder = model.encoder
for param in encoder.parameters():
    param.requires_grad = False




/home/unruffled_satoshi/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/unruffled_satoshi/.local/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/unruffled_satoshi/.local/lib/python3.12/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/unruffled_satoshi/workdir/PFR-ViTCow/videomae_training/VideoMAE/modeling_finetune.py:288: UserWarning: Overwriting vit_small_patch16_224 in regis

Missing keys: ['decoder.blocks.4.norm1.weight', 'decoder.blocks.4.norm1.bias', 'decoder.blocks.4.attn.q_bias', 'decoder.blocks.4.attn.v_bias', 'decoder.blocks.4.attn.qkv.weight', 'decoder.blocks.4.attn.proj.weight', 'decoder.blocks.4.attn.proj.bias', 'decoder.blocks.4.norm2.weight', 'decoder.blocks.4.norm2.bias', 'decoder.blocks.4.mlp.fc1.weight', 'decoder.blocks.4.mlp.fc1.bias', 'decoder.blocks.4.mlp.fc2.weight', 'decoder.blocks.4.mlp.fc2.bias', 'decoder.blocks.5.norm1.weight', 'decoder.blocks.5.norm1.bias', 'decoder.blocks.5.attn.q_bias', 'decoder.blocks.5.attn.v_bias', 'decoder.blocks.5.attn.qkv.weight', 'decoder.blocks.5.attn.proj.weight', 'decoder.blocks.5.attn.proj.bias', 'decoder.blocks.5.norm2.weight', 'decoder.blocks.5.norm2.bias', 'decoder.blocks.5.mlp.fc1.weight', 'decoder.blocks.5.mlp.fc1.bias', 'decoder.blocks.5.mlp.fc2.weight', 'decoder.blocks.5.mlp.fc2.bias', 'decoder.blocks.6.norm1.weight', 'decoder.blocks.6.norm1.bias', 'decoder.blocks.6.attn.q_bias', 'decoder.blocks.6

In [24]:
import os
import torch
from torch.utils.data import Dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import decord  # for video reading
from decord import VideoReader, cpu

class VideoDataset(Dataset):
    def __init__(self, csv_file, video_dir, num_frames=16, transform=None):
        """
        csv_file: path to CSV with columns ['video_name', 'class']
        video_dir: folder containing the video files
        num_frames: number of frames to sample per video
        transform: torchvision-like transforms for frames
        """
        self.video_dir = video_dir
        self.df = pd.read_csv(csv_file)
        
        # Encode string labels as integers
        self.le = LabelEncoder()
        self.df['label_idx'] = self.le.fit_transform(self.df['class'])
        
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = os.path.join(self.video_dir, row['video_path'])
        label = torch.tensor(row['label_idx'], dtype=torch.long)
    
        vr = VideoReader(video_path, ctx=cpu(0))
        total_frames = len(vr)
    
        if total_frames >= self.num_frames:
            indices = torch.linspace(0, total_frames-1, steps=self.num_frames).long()
        else:
            indices = torch.arange(total_frames)
            indices = torch.cat([indices, indices[-1].repeat(self.num_frames - total_frames)])
    
        frames = vr.get_batch(indices)
        frames = torch.from_numpy(frames.asnumpy()).float() / 255.0
        frames = frames.permute(0, 3, 1, 2)  # [T, C, H, W]
    
        if self.transform:
            frames = torch.stack([self.transform(frame) for frame in frames])
    
        return frames, label



In [25]:
from torch.utils.data import DataLoader

train_dataset = VideoDataset(csv_file="/home/unruffled_satoshi/workdir/PFR-ViTCow/videomae_training/data_test/annotations.csv", video_dir="/home/unruffled_satoshi/workdir/PFR-ViTCow/videomae_training/data_test/Damien/")
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)


In [26]:
import torch.nn as nn
import torch.optim as optim

# Freeze encoder
encoder.eval()  # disable dropout/BN
for param in encoder.parameters():
    param.requires_grad = False

# Linear classifier
num_classes = 15  # set your number of classes
linear_head = nn.Linear(encoder.embed_dim, num_classes)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder.to(device)
linear_head.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(linear_head.parameters(), lr=1e-3)


In [28]:
for epoch in range(5):
    linear_head.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    for videos, labels in train_loader:
        # videos: [B, T, C, H, W]
        videos = videos.to(device)
        labels = labels.to(device)
        
        # Forward through frozen encoder
        with torch.no_grad():
            features = encoder(
                videos.permute(0,2,1,3,4),  # [B, C, T, H, W]
                mask=torch.zeros(videos.size(0), encoder.patch_embed.num_patches, dtype=bool, device=device)
            )
            features = features.mean(dim=1)  # average pool over tokens
        
        # Forward through linear classifier
        outputs = linear_head(features)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accuracy
        _, preds = torch.max(outputs, 1)
        running_corrects += (preds == labels).sum().item()
        total_samples += labels.size(0)
        running_loss += loss.item() * labels.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects / total_samples
    print(f"Epoch {epoch+1} done, loss: {epoch_loss:.4f}, accuracy: {epoch_acc:.4f}")


Epoch 1 done, loss: 1.7252, accuracy: 0.3100
Epoch 2 done, loss: 1.7114, accuracy: 0.3260
Epoch 3 done, loss: 1.7232, accuracy: 0.3230
Epoch 4 done, loss: 1.7197, accuracy: 0.3170
Epoch 5 done, loss: 1.6806, accuracy: 0.3230
